### Libraries

In [11]:
pip install numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [12]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time
from pathlib import Path


### Directories

In [13]:
data_dir = Path("..") / "data"
polygon_dir = Path("..") / "polygons"

In [14]:
# create output directories if they do not already exist
figure_dir = Path("..") / "figures"
for dir in [figure_dir]:
    if not dir.is_dir():
        dir.mkdir(parents=True, exist_ok=True)

if not polygon_dir.is_dir():
    from polygonanalyser import xs_all, ys_all, xs_yb, ys_yb


### Video load and show

In [15]:
# load the video
video_path = data_dir / "Video_Test_F2787_125743_01_VIDCKPT_sec.mpg"
cap = cv2.VideoCapture(video_path)

In [16]:
show_video = 0

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
window_name = "flight"

cv2.namedWindow(window_name)

# method for setting frame
def on_trackbar_change(trackbar_value):
    cap.set(cv2.CAP_PROP_POS_FRAMES, trackbar_value)
    return 

cv2.createTrackbar("...", window_name, 0, total_frames - 1, on_trackbar_change)

while cap.isOpened():
    if not show_video: break

    current_trackbar_pos = cv2.getTrackbarPos("...", window_name)

    ret, frame = cap.read()
 
    # if frame is read correctly ret is True. ret will be false when video is over
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    # update the trackbar regularly
    current_frame_id = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
    cv2.setTrackbarPos("...", window_name, current_frame_id)

    # gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow('frame', frame)#, gray)

    # actions to do
    # quit
    if cv2.waitKey(1) == ord('q'):
        break

cv2.destroyAllWindows()

### Mask creation

In [17]:
# set first frame and get the shape
cap.set(cv2.CAP_PROP_POS_FRAMES, 0) 
_, first_frame = cap.read()
frame_shape = first_frame.shape

# define method for determining whether a point is inside a polygon
# https://www.geeksforgeeks.org/dsa/how-to-check-if-a-given-point-lies-inside-a-polygon/
def on_segment(x1, y1, x2, y2, x, y):
    c = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)

    if c != 0:
        return False

    return (min(x1, x2) <= x <= max(x1, x2) and
            min(y1, y2) <= y <= max(y1, y2))

def isInside(arr, x, y):
    n = len(arr)
    inside = False

    j = n - 1

    for i in range(n):
        x1, y1 = arr[i]
        x2, y2 = arr[j]

        # Point lies on the current edge
        if on_segment(x1, y1, x2, y2, x, y):
            return True

        # Check whether the horizontal ray from (x, y)
        # intersects the current edge
        intersect = ((y1 > y) != (y2 > y) and
                     x < (x2 - x1) * (y - y1) / (y2 - y1) + x1)

        if intersect:
            inside = not inside

        j = i

    return inside

# fit the polygons to match position of apparatus in frame
def fit_polygon_all_apparatus(xs, ys):
    # shift polygons to upper right corner
    xs -= np.min(xs)
    ys -= np.min(ys)

    # move to proper startting position
    x_move = frame_shape[1]//2 - 10
    y_move = frame_shape[0]//2 + 85
    xs += x_move 
    ys += y_move 
    return xs, ys

def fit_polygon_yellow_black_apparatus(xs, ys):
    # shift polygons to upper right corner
    xs -= np.min(xs)
    ys -= np.min(ys)

    # move to proper startting position
    x_move = frame_shape[1]//2 + 100
    y_move = frame_shape[0]//2 + 85
    xs += x_move 
    ys += y_move 
    return xs, ys

# load polygons and fit polygons
xs_all = np.load(polygon_dir / "xs_all.npy")
ys_all = np.load(polygon_dir / "ys_all.npy")
xs_yb = np.load(polygon_dir / "xs_yb.npy")
ys_yb = np.load(polygon_dir / "ys_yb.npy")
xs_all, ys_all = fit_polygon_all_apparatus(xs_all, ys_all)
xs_yb, ys_yb = fit_polygon_yellow_black_apparatus(xs_yb, ys_yb)

# convert polygons to list of points
polygon = list(zip(xs_all, ys_all))
polygon = list(zip(xs_yb, ys_yb))

# draw the polygon on the first frame
previous_point = polygon[0]
for point in polygon[1:]:
    cv2.line(first_frame, previous_point, point, (255, 0, 0), 5)
    previous_point = point
cv2.line(first_frame, polygon[-1], polygon[0], (255, 0, 0), 5)

# create a mask using the polygon
mask = np.zeros(frame_shape[:2], dtype=np.uint8)
cv2.fillPoly(
    mask, 
    [np.asarray(polygon, dtype=np.int32)], 
    1
)

# plot the masked image along with the drawing of the polygon
first_frame_masked = first_frame * mask[:, :, None]
cv2.imwrite(Path("..") /"figures" / "image.png", first_frame_masked)




True

### Singular frames 

In [18]:
# define bins used for all histograms
bins = np.arange(0, 256, 10)

def normalize_brightness(frame):
    # normalization using LAB frame
    # https://en.wikipedia.org/wiki/CIELAB_color_space
    # L in LAB is "lightness"
    lab_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab_frame)

    # extract the valid l-channel pixels
    valid_l_pixels = l_channel[mask==1].astype(np.float32)

    # normalize brightness channel by mean=255//2, std=50?
    l_mean, l_std = valid_l_pixels.mean(), valid_l_pixels.std()

    normalized_l = (valid_l_pixels - l_mean) / l_std * 50 + 128
    normalized_l = np.clip(normalized_l, 0, 255).astype(np.uint8)
    l_channel[mask==1] = normalized_l

    # merge back
    normalized_LAB = cv2.merge([l_channel, a_channel, b_channel])
    normalized = cv2.cvtColor(normalized_LAB, cv2.COLOR_LAB2BGR)
    return normalized

def extract_im_and_hist(video_capture, target_frame_index, mask, savefigs=True):
    """
    load a particular frame index in the video capture
    """
    video_capture.set(cv2.CAP_PROP_POS_FRAMES, target_frame_index) # search up the frame
    ret, frame = cap.read()

    if not ret:
        return 

    # filter the frame and normalize it
    frame_filtered = frame * mask[:, :, None]

    # normalize
    frame_filtered = normalize_brightness(frame_filtered)

    # pick out the valid pixels
    valid_pixels = frame_filtered[mask == 1]

    # get counts (density)
    counts_blue, _ = np.histogram(valid_pixels[:, 0], bins, density=False)
    counts_green, _ = np.histogram(valid_pixels[:, 1], bins, density=False)
    counts_red, _ = np.histogram(valid_pixels[:, 2], bins, density=False)

    if savefigs:
        fig, axs=plt.subplots(3, 1)
        axs[0].plot(bins[:-1], counts_blue)
        axs[1].plot(bins[:-1], counts_green)
        axs[2].plot(bins[:-1], counts_red)
        axs[0].set_title("blue", c="blue")
        axs[1].set_title("green", c="green")
        axs[2].set_title("red", c="red")
        fig.tight_layout()

        fig.savefig(Path("..") / "figures" / "histogram.png")
        plt.close()

        # backconvert frame to integers
        
        cv2.imwrite(Path("..") / "figures" / "image.png", frame_filtered)

    return {
        "counts_blue": counts_blue,
        "counts_green": counts_green,
        "counts_red": counts_red,
    }

target_frame = 10000
target_frame = 4000
# target_frame = 1000
extract_im_and_hist(cap, target_frame, mask, savefigs=1)

{'counts_blue': array([   0,    0,    0,    0,  997, 7197, 5959, 4795, 2944, 6987, 7776,
        5309, 4198, 2968, 1560,  559,  187,   92,   13,    1,   20,   19,
           4,    0,    0]),
 'counts_green': array([   0,    0,    0,    0,  205, 6380, 5864, 4718, 2887, 2164, 1627,
         958,  945,  948, 3854, 7609, 5774, 5928, 1609,   69,   22,   19,
           5,    0,    0]),
 'counts_red': array([   0,    0,    0,    6, 3185, 4677, 5563, 3701, 2408, 2059, 1275,
         998,  808,  674,  651,  690,  759, 1365, 6961, 4460, 7059, 3796,
         489,    1,    0])}

In [19]:
target_frame = 10000 - 1000
for addition in range(0, 10000, 10):
    break    
    extract_im_and_hist(cap, target_frame+addition, mask)
    time.sleep(0.05)

In [20]:
target_frame = 0

for addition in range(0, 5000, 100):
    break
    extract_im_and_hist(cap, target_frame+addition, mask=np.ones(frame_shape))
    time.sleep(0.1)


In [21]:
icing_frames = 10000-1000, 10000-1000 + 10000
noicing_frames = 0, 5000

In [22]:
counts = {
    "icing": {
        "counts_blue": [],
        "counts_green": [],
        "counts_red": []
    },
    "noicing": {
        "counts_blue": [],
        "counts_green": [],
        "counts_red": []
    },
}


In [23]:

for i in np.arange(icing_frames[0], icing_frames[1], 100):
    histogram_counts = extract_im_and_hist(cap, i, mask, savefigs=True)
    for key in histogram_counts:
        counts["icing"][key].append(histogram_counts[key])
        # time.sleep(0.1)


In [24]:

for i in np.arange(noicing_frames[0], noicing_frames[1], 100):
    histogram_counts = extract_im_and_hist(cap, i, mask, savefigs=True)
    for key in histogram_counts:
        counts["noicing"][key].append(histogram_counts[key])
        # time.sleep(0.1)

In [25]:
# convert the lists to numpy arrays
for key in counts:
    for color in counts[key]:
        counts[key][color] = np.array(counts[key][color])

In [28]:

# find average histograms and plot them
figicing, axsicing = plt.subplots(3, 1)
for i, key in enumerate(counts["icing"].keys()):
    average_count = counts["icing"][key].mean(axis=0)
    axsicing[i].plot(bins[:-1], average_count)

fignoicing, axsnoicing = plt.subplots(3, 1)
for i, key in enumerate(counts["noicing"].keys()):
    average_count = counts["noicing"][key].mean(axis=0)
    axsnoicing[i].plot(bins[:-1], average_count)

# formatting
for i, color in enumerate(["blue", "green", "red"]):
    axsicing[i].set_title(color, c=color)
    axsnoicing[i].set_title(color, c=color)
figicing.suptitle("Icing")
fignoicing.suptitle("No icing")
figicing.tight_layout()
fignoicing.tight_layout()
figicing.savefig(figure_dir / "Mean icing histogram")
fignoicing.savefig(figure_dir / "Mean no-icing histogram")
plt.close("all")

In [ ]:
import scipy.stats as ss